# Episode 9 — Evaluation First: RAGAS

> *'You can't improve what you don't measure.'*

This episode installs the measurement system we use for the rest of the course.

In [ ]:
import sys,json
from pathlib import Path
cwd=Path().resolve(); repo_root=cwd.parent if cwd.name=='notebooks' else cwd
sys.path.insert(0,str(repo_root/'src'))
from dotenv import load_dotenv; load_dotenv(repo_root/'.env',override=True)
print('ready')

## 1. RAGAS metrics explained

| Metric | Question it answers |
|--------|---------------------|
| **Faithfulness** | Are all claims in the answer grounded in the retrieved context? |
| **Answer Relevance** | Does the answer actually address the question? |
| **Context Precision** | Are the retrieved chunks relevant to the question? |
| **Context Recall** | Does the context contain all info needed to answer? |

In [ ]:
# Install RAGAS
import subprocess,sys
subprocess.run([sys.executable,'-m','pip','install','--quiet','ragas','datasets'],check=True)
print('✅ RAGAS installed')

In [ ]:
# Build the pipeline components
from rag.retrieval.vector_retriever import VectorRetriever
from rag.chains.rag_chain import build_rag_chain
retriever = VectorRetriever()
chain     = build_rag_chain(retriever)
print('Pipeline ready')

In [ ]:
# Load our 30-question test set
import json
from pathlib import Path
test_set_path = repo_root/'src'/'rag'/'evaluation'/'test_set'/'questions.json'
if test_set_path.exists():
    questions = json.loads(test_set_path.read_text())
    print(f'Loaded {len(questions)} test questions')
    from collections import Counter
    for qtype,n in Counter(q['type'] for q in questions).most_common():
        print(f'  {qtype:<20} {n}')
else:
    print('Run Episode 2 notebook first to generate questions.json')

In [ ]:
# Run RAGAS on a small sample (3 questions) to keep API costs low during demo
# Full 30-question run: python scripts/eval.py
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from datasets import Dataset

sample_qs = questions[:3]  # use 3 for demo
data = {'question':[],'answer':[],'contexts':[],'ground_truth':[]}
for q in sample_qs:
    docs     = retriever.retrieve(q['question'])
    contexts = [d.page_content for d in docs]
    answer   = chain.invoke({'question':q['question']})
    data['question'].append(q['question'])
    data['answer'].append(answer if isinstance(answer,str) else str(answer))
    data['contexts'].append(contexts)
    data['ground_truth'].append(q.get('ground_truth',''))

dataset = Dataset.from_dict(data)
result  = evaluate(dataset, metrics=[faithfulness,answer_relevancy,context_precision,context_recall])
print('\nRAGAS Scores (Phase 1 baseline — 3 questions):')
for k,v in result.items():
    if isinstance(v,(int,float)): print(f'  {k:<25} {float(v):.4f}')

## Next: Episode 10 — Reranking

The single biggest quality upgrade. We add Cohere Rerank after vector search
and re-run RAGAS to see the improvement.